# FRA371 Kinematics Labs: Symbolic Kinematics Calculator (FK, IK, DK)

This notebook is an end-to-end **Symbolic and Numerical Kinematics Calculator** for the 3-DOF robot manipulator (`my_robot` from FRA371 Lab 1 and Lab 2).

### Learning Objectives:
1. **Forward Kinematics (FK)**:
   - Derive link transformation matrices symbolically using **Craig's Modified Denavit-Hartenberg (MDH)** convention.
   - Obtain closed-form expressions for end-effector position $\mathbf{p}(q)$ and rotation matrix $\mathbf{R}(q)$.
   - Validate against the Robotics Toolbox MDH model (`robot_mdh`) and the URDF model (`urdf_robot`).

2. **Inverse Kinematics (IK)**:
   - Derive analytical closed-form solutions for joint variables $(q_1, q_2, q_3)$ from a target position $\mathbf{M} = [x, y, z]^T$.
   - Formulate and solve for both kinematic branches (Branch 1: Front / Forward, Branch 2: Back / Flipped).
   - Verify solutions using forward kinematics round-trip and compare with numerical IK (`ikine_LM`).

3. **Differential Kinematics (DK)**:
   - Symbolically derive the **Position Jacobian** $\mathbf{J}_v(q) = \frac{\partial \mathbf{p}}{\partial q}$.
   - Symbolically derive the **Angular Jacobian** $\mathbf{J}_\omega(q)$ and the full $6 \times 3$ geometric Jacobian $\mathbf{J}_0(q)$.
   - Perform **Singularity Analysis** by evaluating $\det(\mathbf{J}_v(q))$.
   - Obtain the symbolic **Inverse Jacobian** $\mathbf{J}_v^{-1}(q)$ for resolved-rate motion control ($\\dot{q} = \mathbf{J}_v^{-1} \dot{\mathbf{p}}$).
   - Validate against Robotics Toolbox `jacob0(q)`.

4. **Unified Interactive Calculator**:
   - Provide a clean, reusable Python calculator class `RobotKinematicsCalculator` that executes FK, IK, and DK on arbitrary user inputs.


## 1. Environment Setup & Robot Loading

The robot has 3 degrees of freedom configured in an **RRP (Revolute-Revolute-Prismatic)** architecture:
- **Joint 1 ($q_1$)**: Revolute about base vertical axis $Z_0$, limits: $[-\pi/2, \pi/2]$ rad ($[-90^\circ, 90^\circ]$).
- **Joint 2 ($q_2$)**: Revolute about horizontal axis, limits: $[-\pi/2, \pi/2]$ rad ($[-90^\circ, 90^\circ]$).
- **Joint 3 ($q_3$)**: Prismatic along the arm axis, limits: $[0.0, 0.40]$ m.
- **Link Lengths**:
  - $L_1 = 0.50$ m (Height from base to Joint 2)
  - $L_2 = 0.10$ m (Prismatic arm structural offset)
  - $L_3 = 0.40$ m (End-effector tool distance)


In [20]:
import numpy as np
import sympy as sp
from pathlib import Path

# Spatial math & robotics toolbox
from spatialmath import SE3
from spatialmath.base import trinv
from roboticstoolbox import Robot, DHRobot, RevoluteMDH, PrismaticMDH
import spatialgeometry as sg
import swift

np.set_printoptions(suppress=True, precision=4)

# 1. Load URDF Robot Model
HERE = Path.cwd().as_posix()
if hasattr(Robot, "URDF_read"):             # older roboticstoolbox (<1.5.0)
    links, name = Robot.URDF_read("my_robot/robot.urdf", tld=HERE)[:2]
else:                                       # roboticstoolbox 1.5.0+
    from roboticstoolbox.models.URDF import URDFRobot
    from xacrodoc.packages import update_package_cache
    update_package_cache({"my_robot": HERE + "/my_robot"})
    links, name = URDFRobot.URDF_read(HERE + "/my_robot/robot.urdf")[:2]

urdf_robot = Robot(links, name=name)
print(f"[OK] Loaded URDF robot '{name}' with {urdf_robot.n} joints.")

# 2. Build Craig's Modified DH (MDH) Model
PI_DIV_2 = np.pi / 2
L1 = 0.50
L2 = 0.10
L3 = 0.40

robot_mdh = DHRobot([
    RevoluteMDH(    a=0.0, alpha=0.0,      d=L1,  offset=0.0),
    RevoluteMDH(    a=0.0, alpha=PI_DIV_2, d=0.0, offset=PI_DIV_2),
    PrismaticMDH(   a=0.0, alpha=PI_DIV_2, theta=0.0, offset=L2)
], name='my_mdh')
robot_mdh.tool = SE3(0.0, 0.0, L3)

print("[OK] Loaded MDH Model:")
print(robot_mdh)

# 3. Optional Swift Simulator
try:
    import IPython
    if IPython.get_ipython() is not None:
        env = swift.Swift()
        env.launch(realtime=True, browser="notebook")
        env.add(urdf_robot)
        urdf_robot.q = [0.0, 0.0, 0.0]
        env.step()
        print("[OK] Swift simulator connected.")
    else:
        print("[Notice] Non-interactive environment detected; skipping Swift browser launch.")
except Exception as e:
    print(f"[Notice] Swift simulator not opened in standalone mode ({e}). Visualizer optional.")


[OK] Loaded URDF robot 'my_robot' with 3 joints.
[OK] Loaded MDH Model:
DHRobot: my_mdh, 3 joints (RRP), modified DH parameters
┌──────┬───────┬───────────┬───────────┐
│ aⱼ₋₁ │ ⍺ⱼ₋₁  │    θⱼ     │    dⱼ     │
├──────┼───────┼───────────┼───────────┤
│  0.0 │  0.0° │        q1 │       0.5 │
│  0.0 │ 90.0° │  q2 + 90° │       0.0 │
│  0.0 │ 90.0° │      0.0° │  q3 + 0.1 │
└──────┴───────┴───────────┴───────────┘

┌──────┬─────────────────────────────────────┐
│ tool │ t = 0, 0, 0.4; rpy/xyz = 0°, 0°, 0° │
└──────┴─────────────────────────────────────┘



C:\Users\User\AppData\Local\Temp\ipykernel_36848\4132404966.py:48: FutureWarning: launch(browser="notebook") is deprecated, use launch(notebook=True)
  env.launch(realtime=True, browser="notebook")


[OK] Swift simulator connected.


In [22]:
# Cross-check MDH Model vs URDF Model across range of motion
lo, hi = urdf_robot.qlim
test_configs = [
    np.array([0.0, 0.0, 0.0]),
    np.array([np.deg2rad(30), 0.0, 0.6 * 0.4]),
    np.array([np.deg2rad(-25), 0.0, 0.45 * 0.4]),
    np.array([np.deg2rad(45), np.deg2rad(30), 0.20]),
    np.array([np.deg2rad(-45), np.deg2rad(-30), 0.35]),
]

print("=== Cross-Checking MDH Model vs URDF Model ===")
max_err = 0.0
for q in test_configs:
    T_mdh = robot_mdh.fkine(q)
    T_urdf = urdf_robot.fkine(q)
    pos_err = np.linalg.norm(T_mdh.t - T_urdf.t)
    rot_err = np.linalg.norm(T_mdh.R - T_urdf.R)
    max_err = max(max_err, pos_err)
    print(f"q = {np.round(q, 3)} | Pos Diff = {pos_err:.2e} m | Rot Diff = {rot_err:.2e}")

assert max_err < 1e-4, "Mismatch between URDF and MDH models!"
print(f"-> All tests passed! Maximum discrepancy: {max_err:.2e} m (MDH vs URDF: OK)\n")


=== Cross-Checking MDH Model vs URDF Model ===
q = [0. 0. 0.] | Pos Diff = 2.45e-17 m | Rot Diff = 1.22e-16
q = [0.524 0.    0.24 ] | Pos Diff = 1.11e-16 m | Rot Diff = 1.17e-16
q = [-0.436  0.     0.18 ] | Pos Diff = 0.00e+00 m | Rot Diff = 1.17e-16
q = [0.785 0.524 0.2  ] | Pos Diff = 1.11e-16 m | Rot Diff = 2.29e-16
q = [-0.785 -0.524  0.35 ] | Pos Diff = 8.33e-17 m | Rot Diff = 5.55e-17
-> All tests passed! Maximum discrepancy: 1.11e-16 m (MDH vs URDF: OK)



## 2. Forward Kinematics (FK) - Symbolic Derivation & Analysis

### Craig's Modified Denavit-Hartenberg (MDH) Formulation
Under Craig's Modified DH convention, each link transformation $^{i-1}\mathbf{T}_i$ is defined by:
$${}^{i-1}\mathbf{T}_i = \text{Rot}_{X_{i-1}}(\alpha_{i-1}) \cdot \text{Trans}_{X_{i-1}}(a_{i-1}) \cdot \text{Rot}_{Z_i}(\theta_i) \cdot \text{Trans}_{Z_i}(d_i)$$

The resulting homogeneous transformation matrix is:
$${}^{i-1}\mathbf{T}_i = \begin{bmatrix}
\cos\theta_i & -\sin\theta_i & 0 & a_{i-1} \\
\sin\theta_i\cos\alpha_{i-1} & \cos\theta_i\cos\alpha_{i-1} & -\sin\alpha_{i-1} & -d_i\sin\alpha_{i-1} \\
\sin\theta_i\sin\alpha_{i-1} & \cos\theta_i\sin\alpha_{i-1} & \cos\alpha_{i-1} & d_i\cos\alpha_{i-1} \\
0 & 0 & 0 & 1
\end{bmatrix}$$

### Modified DH Parameter Table for `my_robot`:
| Frame $i$ | $\alpha_{i-1}$ | $a_{i-1}$ | $\theta_i$ | $d_i$ | Type | Variable |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| **1** | $0$ | $0$ | $q_1$ | $L_1 = 0.50$ | Revolute | $q_1$ |
| **2** | $\pi/2$ | $0$ | $q_2 + \pi/2$ | $0$ | Revolute | $q_2$ |
| **3** | $\pi/2$ | $0$ | $0$ | $q_3 + L_2$ | Prismatic | $q_3$ |
| **Tool (E)** | $0$ | $0$ | $0$ | $L_3 = 0.40$ | Fixed offset along $Z_3$ | - |

The overall transformation from base frame $\{0\}$ to end-effector frame $\{E\}$ is:
$${}^0\mathbf{T}_E = {}^0\mathbf{T}_1(q_1) \cdot {}^1\mathbf{T}_2(q_2) \cdot {}^2\mathbf{T}_3(q_3) \cdot {}^3\mathbf{T}_E$$


In [23]:
def symbolic_fk(q=None, L=(0.50, 0.10, 0.40), verbose=True):
    """
    Symbolically derives Forward Kinematics (FK) for the 3-DOF RRP robot
    using Craig's Modified DH parameters, with optional numerical evaluation.
    
    Parameters:
        q: array-like of 3 joint values [q1 (rad), q2 (rad), q3 (m)], or None
        L: tuple of link lengths (L1, L2, L3)
        verbose: bool, print formatted symbolic expressions
        
    Returns:
        T0E: SymPy Matrix (4x4 overall symbolic transformation)
        pos_expr: SymPy Matrix (3x1 symbolic position vector)
        rot_expr: SymPy Matrix (3x3 symbolic rotation matrix)
        (pos_num, T_num): if q is provided, numerical position and matrix
    """
    # 1. Define symbolic variables
    q1, q2, q3 = sp.symbols('q1 q2 q3', real=True)
    L1, L2, L3 = sp.symbols('L1 L2 L3', real=True)

    # 2. Craig Modified DH transformation matrix formula
    def mdh_matrix(alpha, a, theta, d):
        return sp.Matrix([
            [sp.cos(theta), -sp.sin(theta), 0, a],
            [sp.sin(theta)*sp.cos(alpha),  sp.cos(theta)*sp.cos(alpha), -sp.sin(alpha), -d*sp.sin(alpha)],
            [sp.sin(theta)*sp.sin(alpha),  sp.cos(theta)*sp.sin(alpha),  sp.cos(alpha),  d*sp.cos(alpha)],
            [0, 0, 0, 1]
        ])

    # 3. Individual link transformations
    T01 = mdh_matrix(0, 0, q1, L1)
    T12 = mdh_matrix(sp.pi/2, 0, q2 + sp.pi/2, 0)
    T23 = mdh_matrix(sp.pi/2, 0, 0, q3 + L2)
    T3E = sp.Matrix([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, L3],
        [0, 0, 0, 1]
    ])

    # 4. Chain multiplication & trigonometric simplification
    T02 = sp.trigsimp(T01 * T12)
    T03 = sp.trigsimp(T02 * T23)
    T0E = sp.trigsimp(T03 * T3E)

    pos_expr = T0E[:3, 3]
    rot_expr = T0E[:3, :3]

    if verbose:
        print("="*60)
        print("SYMBOLIC FORWARD KINEMATICS MATRIX T_0E:")
        print("="*60)
        sp.pprint(T0E)
        print("\n" + "="*60)
        print("END-EFFECTOR POSITION EQUATIONS [x, y, z]^T:")
        print("="*60)
        print(f"x = {pos_expr[0]}")
        print(f"y = {pos_expr[1]}")
        print(f"z = {pos_expr[2]}")
        print("\n" + "="*60)
        print("END-EFFECTOR ORIENTATION MATRIX R_0E:")
        print("="*60)
        sp.pprint(rot_expr)

    # 5. Evaluate numerically if q is provided
    if q is not None:
        subs = {q1: q[0], q2: q[1], q3: q[2], L1: L[0], L2: L[1], L3: L[2]}
        pos_num = np.array([float(val.subs(subs)) for val in pos_expr])
        T_num = np.array(T0E.subs(subs)).astype(np.float64)
        return T0E, pos_expr, rot_expr, pos_num, T_num

    return T0E, pos_expr, rot_expr


In [24]:
# Run symbolic derivation and evaluate at test points
T_sym, pos_sym, rot_sym = symbolic_fk(verbose=True)

print("\n" + "#"*60)
print("VERIFICATION: Symbolic FK vs Robotics Toolbox MDH vs URDF")
print("#"*60)

test_points = {
    "q_Home": np.array([0.0, 0.0, 0.0]),
    "q_A (Lab 1)": np.array([np.deg2rad(30), 0.0, 0.6 * 0.4]),
    "q_B (Lab 1)": np.array([np.deg2rad(-25), 0.0, 0.45 * 0.4]),
    "q_Arbitrary": np.array([0.45, -0.30, 0.25])
}

for name, q_val in test_points.items():
    _, _, _, p_sym, T_num = symbolic_fk(q=q_val, verbose=False)
    T_mdh = robot_mdh.fkine(q_val)
    T_urdf = urdf_robot.fkine(q_val)

    pos_err_mdh = np.linalg.norm(p_sym - T_mdh.t)
    pos_err_urdf = np.linalg.norm(p_sym - T_urdf.t)
    rot_err_mdh = np.linalg.norm(T_num[:3, :3] - T_mdh.R)

    print(f"[{name}] Joint Angles: {np.round(q_val, 4)}")
    print(f"  Symbolic Tip Pos : {np.round(p_sym, 4)}")
    print(f"  MDH Tip Pos      : {np.round(T_mdh.t, 4)}")
    print(f"  URDF Tip Pos     : {np.round(T_urdf.t, 4)}")
    print(f"  Pos Error (Sym - MDH) : {pos_err_mdh:.2e} m")
    print(f"  Pos Error (Sym - URDF): {pos_err_urdf:.2e} m")
    print(f"  Rot Error (Sym - MDH) : {rot_err_mdh:.2e}")
    print("-" * 50)


SYMBOLIC FORWARD KINEMATICS MATRIX T_0E:
⎡-sin(q₂)⋅cos(q₁)  sin(q₁)   cos(q₁)⋅cos(q₂)    (L₂ + L₃ + q₃)⋅cos(q₁)⋅cos(q₂) ↪
⎢                                                                              ↪
⎢-sin(q₁)⋅sin(q₂)  -cos(q₁)  sin(q₁)⋅cos(q₂)    (L₂ + L₃ + q₃)⋅sin(q₁)⋅cos(q₂) ↪
⎢                                                                              ↪
⎢    cos(q₂)          0          sin(q₂)      L₁ + L₃⋅sin(q₂) + (L₂ + q₃)⋅sin( ↪
⎢                                                                              ↪
⎣       0             0             0                          1               ↪

↪    ⎤
↪    ⎥
↪    ⎥
↪    ⎥
↪ q₂)⎥
↪    ⎥
↪    ⎦

END-EFFECTOR POSITION EQUATIONS [x, y, z]^T:
x = (L2 + L3 + q3)*cos(q1)*cos(q2)
y = (L2 + L3 + q3)*sin(q1)*cos(q2)
z = L1 + L3*sin(q2) + (L2 + q3)*sin(q2)

END-EFFECTOR ORIENTATION MATRIX R_0E:
⎡-sin(q₂)⋅cos(q₁)  sin(q₁)   cos(q₁)⋅cos(q₂)⎤
⎢                                           ⎥
⎢-sin(q₁)⋅sin(q₂)  -cos(q₁)  sin(q₁)⋅cos(q₂)⎥
⎢        

## 3. Inverse Kinematics (IK) - Symbolic Algebraic Derivation

Given a desired end-effector target position $\mathbf{M} = [x, y, z]^T$, we solve for the joint variables $(q_1, q_2, q_3)$ such that $\mathbf{p}(q_1, q_2, q_3) = \mathbf{M}$.

From the Forward Kinematics derived in Section 2:
$$\begin{aligned}
x &= (L_2 + L_3 + q_3) \cos(q_1) \cos(q_2) \qquad &(1) \\
y &= (L_2 + L_3 + q_3) \sin(q_1) \cos(q_2) \qquad &(2) \\
z &= L_1 + (L_2 + L_3 + q_3) \sin(q_2) \qquad &(3)
\end{aligned}$$

Let $R = L_2 + L_3 + q_3$ be the radial extension distance along the arm ($R > 0$).

---

### Step A: Solve for $q_3$ (Radial Extension / Sphere Equation)
Sum the squares of $(1)$, $(2)$, and $(z - L_1)^2$:
$$\begin{aligned}
x^2 + y^2 + (z - L_1)^2 &= R^2 \cos^2(q_1)\cos^2(q_2) + R^2 \sin^2(q_1)\cos^2(q_2) + R^2 \sin^2(q_2) \\
&= R^2 \cos^2(q_2) \big[\cos^2(q_1) + \sin^2(q_1)\big] + R^2 \sin^2(q_2) \\
&= R^2 \big[\cos^2(q_2) + \sin^2(q_2)\big] \\
&= R^2
\end{aligned}$$

Taking the positive square root (since $R = L_2 + L_3 + q_3 > 0$):
$$R = +\sqrt{x^2 + y^2 + (z - L_1)^2}$$
$$\boxed{q_3 = \sqrt{x^2 + y^2 + (z - L_1)^2} - L_2 - L_3}$$

---

### Step B: Solve for $q_1$ (Base Azimuth Angle)
Divide equation $(2)$ by equation $(1)$:
$$\frac{y}{x} = \frac{R \sin(q_1)\cos(q_2)}{R \cos(q_1)\cos(q_2)} = \tan(q_1)$$

Using the four-quadrant `atan2` function yields two kinematically valid branches:
- **Branch 1 (Forward / Direct reach)**:
  $$\boxed{q_{1,\text{b1}} = \text{atan2}(y, x)}$$
- **Branch 2 (Flipped / Backward reach)**:
  $$\boxed{q_{1,\text{b2}} = \text{atan2}(-y, -x) = q_{1,\text{b1}} \pm \pi}$$

---

### Step C: Solve for $q_2$ (Elevation Angle)
Project the target onto the horizontal ground plane:
$$r_{xy} = \sqrt{x^2 + y^2} = |R \cos(q_2)|$$

From equation $(3)$, $(z - L_1) = R \sin(q_2)$. Using $\text{atan2}(\text{vertical}, \text{horizontal})$:
- For **Branch 1** (pointing towards $(x, y)$):
  $$\boxed{q_{2,\text{b1}} = \text{atan2}(z - L_1, r_{xy}) = \text{atan2}(z - L_1, \sqrt{x^2 + y^2})}$$
- For **Branch 2** (facing away, arm rotated backward over the top):
  $$\boxed{q_{2,\text{b2}} = \text{atan2}(z - L_1, -r_{xy}) = \text{atan2}(z - L_1, -\sqrt{x^2 + y^2})}$$


In [25]:
def symbolic_ik(M=None, L=(0.50, 0.10, 0.40), check_limits=True, verbose=True):
    """
    Algebraically derives Inverse Kinematics for the 3-DOF RRP robot
    from closed-form FK equations using SymPy elimination.
    
    Parameters:
        M: array-like of 3 target coordinates [x, y, z] in base frame, or None
        L: tuple of link lengths (L1, L2, L3)
        check_limits: bool, verify if solutions lie within robot joint limits
        verbose: bool, display derivation steps
        
    Returns:
        sym_dict: dictionary containing symbolic expressions for both branches
        b1_num, b2_num: numerical joint solution arrays if M is provided
    """
    # 1. Define symbolic variables
    x, y, z = sp.symbols('x y z', real=True)
    q1, q2, q3 = sp.symbols('q1 q2 q3', real=True)
    L1, L2, L3 = sp.symbols('L1 L2 L3', real=True)

    # 2. Closed-form FK equations
    R_ext = L2 + L3 + q3
    fx = R_ext * sp.cos(q1) * sp.cos(q2)
    fy = R_ext * sp.sin(q1) * sp.cos(q2)
    fz = L1 + R_ext * sp.sin(q2)

    # 3. Elimination for q3: x^2 + y^2 + (z - L1)^2
    sum_lhs = x**2 + y**2 + (z - L1)**2
    sum_rhs = sp.trigsimp(fx**2 + fy**2 + (fz - L1)**2)  # simplifies to (L2 + L3 + q3)**2
    eq_elim_q3 = sp.Eq(sum_lhs, sum_rhs)

    q3_sols = sp.solve(eq_elim_q3, q3)
    # The physically positive extension solution
    q3_expr = q3_sols[1]  # sqrt(x**2 + y**2 + (z - L1)**2) - L2 - L3

    # 4. Elimination for q1: fy / fx = tan(q1)
    tan_q1 = sp.trigsimp(fy / fx)
    q1_b1 = sp.atan2(y, x)
    q1_b2 = sp.atan2(-y, -x)

    # 5. Elimination for q2
    r_xy = sp.sqrt(x**2 + y**2)
    q2_b1 = sp.atan2(z - L1, r_xy)
    q2_b2 = sp.atan2(z - L1, -r_xy)

    sym_dict = {
        "branch1": (q1_b1, q2_b1, q3_expr),
        "branch2": (q1_b2, q2_b2, q3_expr),
        "equations": {
            "eq_x": sp.Eq(x, fx),
            "eq_y": sp.Eq(y, fy),
            "eq_z": sp.Eq(z, fz),
            "elim_sphere": eq_elim_q3,
            "q3_sol": q3_expr,
            "tan_q1": sp.Eq(sp.Symbol("y/x"), tan_q1),
        }
    }

    if verbose:
        print("="*60)
        print("SYMBOLIC INVERSE KINEMATICS EQUATIONS:")
        print("="*60)
        print(f"Spherical elimination: {sum_lhs} = {sum_rhs}")
        print(f"q3 expression        : {q3_expr}")
        print(f"Branch 1 q1 (atan2)  : {q1_b1}")
        print(f"Branch 1 q2 (atan2)  : {q2_b1}")
        print(f"Branch 2 q1 (atan2)  : {q1_b2}")
        print(f"Branch 2 q2 (atan2)  : {q2_b2}")
        print("="*60)

    # 6. Evaluate numerically if M is given
    if M is not None:
        subs = {x: M[0], y: M[1], z: M[2], L1: L[0], L2: L[1], L3: L[2]}
        b1_num = np.array([float(expr.subs(subs)) for expr in sym_dict["branch1"]])
        b2_num = np.array([float(expr.subs(subs)) for expr in sym_dict["branch2"]])

        if check_limits:
            qlim = urdf_robot.qlim
            for b_idx, (b_name, b_val) in enumerate([("Branch 1", b1_num), ("Branch 2", b2_num)]):
                in_limit = np.all((b_val >= qlim[0]) & (b_val <= qlim[1]))
                status = "VALID (Inside Limits)" if in_limit else "EXCEEDS LIMITS"
                if verbose:
                    print(f"{b_name}: {np.round(b_val, 4)} -> {status}")

        return sym_dict, b1_num, b2_num

    return sym_dict


In [26]:
# Target computation from Camera frame to Base frame (Lab 2 Part A & B)
# Given from camera perception:
Cam_T_Object = np.array([
    [0.0, 0.0, 0.0, 0.0347],
    [0.0, 0.0, 0.0, 0.0563],
    [0.0, 0.0, 0.0, 0.9768],
    [0.0, 0.0, 0.0, 1.0000]
])

Cam_T_Base = np.array([
    [ 0.9955, -0.0949,  0.0003, -0.4325],
    [-0.0947, -0.9942, -0.0518, -0.0507],
    [ 0.0052,  0.0515, -0.9987,  1.9335],
    [ 0.0000,  0.0000,  0.0000,  1.0000]
])

# Base_T_Object = (Cam_T_Base)^(-1) * Cam_T_Object
Base_T_Object = np.round(trinv(Cam_T_Base) @ Cam_T_Object, 4)
M = Base_T_Object[:3, 3]

print(f"Computed Target in Base Frame M: {M}")

# Solve IK symbolically and evaluate
sym_ik_res, branch1, branch2 = symbolic_ik(M=M, verbose=True)

# Select valid branch inside joint limits
qlim = urdf_robot.qlim
b1_valid = np.all((branch1 >= qlim[0]) & (branch1 <= qlim[1]))
b2_valid = np.all((branch2 >= qlim[0]) & (branch2 <= qlim[1]))

q_goal = branch1 if b1_valid else branch2
print(f"\nChosen IK Joint Configuration q_goal: {np.round(q_goal, 4)}")


Computed Target in Base Frame M: [ 0.45   -0.2     0.9501]
SYMBOLIC INVERSE KINEMATICS EQUATIONS:
Spherical elimination: x**2 + y**2 + (-L1 + z)**2 = (L2 + L3 + q3)**2
q3 expression        : -L2 - L3 + sqrt(L1**2 - 2*L1*z + x**2 + y**2 + z**2)
Branch 1 q1 (atan2)  : atan2(y, x)
Branch 1 q2 (atan2)  : atan2(-L1 + z, sqrt(x**2 + y**2))
Branch 2 q1 (atan2)  : atan2(-y, -x)
Branch 2 q2 (atan2)  : atan2(-L1 + z, -sqrt(x**2 + y**2))
Branch 1: [-0.4182  0.7405  0.1672] -> VALID (Inside Limits)
Branch 2: [2.7234 2.4011 0.1672] -> EXCEEDS LIMITS

Chosen IK Joint Configuration q_goal: [-0.4182  0.7405  0.1672]


In [27]:
# Cross-check IK solutions with Forward Kinematics and Numerical Toolbox IK
print("=== Cross-Checking Inverse Kinematics Solutions ===")

for i, qb in enumerate([branch1, branch2], 1):
    # Round-trip through URDF and MDH Forward Kinematics
    p_urdf = urdf_robot.fkine(qb).t
    p_mdh = robot_mdh.fkine(qb).t
    _, _, _, p_sym, _ = symbolic_fk(q=qb, verbose=False)

    err_target = np.linalg.norm(p_sym - M)
    print(f"Branch {i}: q = {np.round(qb, 4)}")
    print(f"  Reached Position (Symbolic) : {np.round(p_sym, 4)}")
    print(f"  Reached Position (URDF)     : {np.round(p_urdf, 4)}")
    print(f"  Position Error vs Target M  : {err_target:.2e} m (Must be < 0.03 m)")
    print("-" * 55)

# Comparison with Numerical IK (Levenberg-Marquardt)
sol = urdf_robot.ikine_LM(SE3(*M), q0=q_goal, mask=[1, 1, 1, 0, 0, 0], joint_limits=True)
print("\nComparison with Numerical Toolbox IK:")
print(f"  Hand / Symbolic q_goal : {np.round(q_goal, 4)}")
print(f"  Numerical ikine_LM q   : {np.round(sol.q, 4)} | Converged: {sol.success}")
print(f"  Difference             : {np.linalg.norm(q_goal - sol.q):.2e}")


=== Cross-Checking Inverse Kinematics Solutions ===
Branch 1: q = [-0.4182  0.7405  0.1672]
  Reached Position (Symbolic) : [ 0.45   -0.2     0.9501]
  Reached Position (URDF)     : [ 0.45   -0.2     0.9501]
  Position Error vs Target M  : 1.36e-16 m (Must be < 0.03 m)
-------------------------------------------------------
Branch 2: q = [2.7234 2.4011 0.1672]
  Reached Position (Symbolic) : [ 0.45   -0.2     0.9501]
  Reached Position (URDF)     : [ 0.45   -0.2     0.9501]
  Position Error vs Target M  : 2.08e-16 m (Must be < 0.03 m)
-------------------------------------------------------

Comparison with Numerical Toolbox IK:
  Hand / Symbolic q_goal : [-0.4182  0.7405  0.1672]
  Numerical ikine_LM q   : [-0.4182  0.7405  0.1672] | Converged: True
  Difference             : 0.00e+00


## 4. Differential Kinematics (DK) - Symbolic Jacobian & Singularities

Differential Kinematics maps the joint velocities $\dot{\mathbf{q}} = [\dot{q}_1, \dot{q}_2, \dot{q}_3]^T$ to the operational space velocities of the end-effector:
$$\mathbf{v}_e = \begin{bmatrix} \dot{\mathbf{p}}_e \\ \boldsymbol{\omega}_e \end{bmatrix} = \mathbf{J}_0(q) \dot{\mathbf{q}}$$

where $\mathbf{J}_0(q) \in \mathbb{R}^{6 \times 3}$ is the geometric Jacobian:
$$\mathbf{J}_0(q) = \begin{bmatrix} \mathbf{J}_v(q) \\ \mathbf{J}_\omega(q) \end{bmatrix}$$

---

### 1. Position / Linear Velocity Jacobian $\mathbf{J}_v(q)$
By the chain rule, $\dot{\mathbf{p}}_e = \frac{\partial \mathbf{p}_e}{\partial \mathbf{q}} \dot{\mathbf{q}}$:
$$\mathbf{J}_v(q) = \begin{bmatrix}
\frac{\partial x}{\partial q_1} & \frac{\partial x}{\partial q_2} & \frac{\partial x}{\partial q_3} \\[6pt]
\frac{\partial y}{\partial q_1} & \frac{\partial y}{\partial q_2} & \frac{\partial y}{\partial q_3} \\[6pt]
\frac{\partial z}{\partial q_1} & \frac{\partial z}{\partial q_2} & \frac{\partial z}{\partial q_3}
\end{bmatrix}$$

With $R = L_2 + L_3 + q_3$:
$$\mathbf{J}_v(q) = \begin{bmatrix}
-R \sin(q_1)\cos(q_2) & -R \cos(q_1)\sin(q_2) & \cos(q_1)\cos(q_2) \\
 R \cos(q_1)\cos(q_2) & -R \sin(q_1)\sin(q_2) & \sin(q_1)\cos(q_2) \\
 0 & R \cos(q_2) & \sin(q_2)
\end{bmatrix}$$

---

### 2. Angular Velocity Jacobian $\mathbf{J}_\omega(q)$
For each joint $i$:
- If joint $i$ is revolute: $\mathbf{J}_{\omega, i} = \mathbf{z}_{i-1}$ (axis of rotation in base frame).
- If joint $i$ is prismatic: $\mathbf{J}_{\omega, i} = \mathbf{0}_{3 \times 1}$.

For our robot:
- Joint 1 (Revolute about $Z_0$): $\mathbf{z}_0 = [0, 0, 1]^T$
- Joint 2 (Revolute about $Z_1$): In base frame, $\mathbf{z}_1 = [\sin(q_1), -\cos(q_1), 0]^T$
- Joint 3 (Prismatic along $Z_2$): $\mathbf{0}_{3 \times 1}$

$$\mathbf{J}_\omega(q) = \begin{bmatrix}
0 & \sin(q_1) & 0 \\
0 & -\cos(q_1) & 0 \\
1 & 0 & 0
\end{bmatrix}$$

---

### 3. Singularity Analysis via $\det(\mathbf{J}_v)$
Taking the determinant of the $3 \times 3$ linear Jacobian:
$$\det\big(\mathbf{J}_v(q)\big) = (L_2 + L_3 + q_3)^2 \cos(q_2)$$

Singular configurations occur when $\det(\mathbf{J}_v) = 0$:
1. **Radial Boundary Singularity**:
   $$L_2 + L_3 + q_3 = 0 \implies q_3 = -(L_2 + L_3)$$
   Since $q_3 \ge 0$, this is outside the physical joint range and cannot be reached.
2. **Wrist / Elevation Singularity (Spherical Singularity)**:
   $$\cos(q_2) = 0 \iff q_2 = \pm \frac{\pi}{2} = \pm 90^\circ$$
   When the arm points straight up or straight down along the $Z_0$ axis, joint 1 and joint 2 become linearly dependent; velocity in the horizontal plane perpendicular to the arm is lost.

---

### 4. Analytical Inverse Jacobian $\mathbf{J}_v^{-1}(q)$
For resolved-rate velocity control ($\dot{\mathbf{q}} = \mathbf{J}_v^{-1} \dot{\mathbf{p}}$):
$$\mathbf{J}_v^{-1}(q) = \begin{bmatrix}
-\frac{\sin(q_1)}{R \cos(q_2)} & \frac{\cos(q_1)}{R \cos(q_2)} & 0 \\[6pt]
-\frac{\cos(q_1)\sin(q_2)}{R} & -\frac{\sin(q_1)\sin(q_2)}{R} & \frac{\cos(q_2)}{R} \\[6pt]
\cos(q_1)\cos(q_2) & \sin(q_1)\cos(q_2) & \sin(q_2)
\end{bmatrix}$$
Notice the division by $\cos(q_2)$, confirming that $q_2 = \pm 90^\circ$ is a true physical singularity.


In [28]:
def symbolic_dk(q=None, q_dot=None, L=(0.50, 0.10, 0.40), verbose=True):
    """
    Symbolically derives Differential Kinematics (DK), the geometric Jacobian,
    determinant, and analytical inverse Jacobian for the 3-DOF RRP robot.
    
    Parameters:
        q: array-like of 3 joint positions [q1, q2, q3], or None
        q_dot: array-like of 3 joint velocities [q1_dot, q2_dot, q3_dot], or None
        L: tuple of link lengths (L1, L2, L3)
        verbose: bool, print formatted symbolic expressions
        
    Returns:
        J_geom: SymPy Matrix (6x3 full geometric Jacobian)
        J_v: SymPy Matrix (3x3 linear velocity Jacobian)
        J_w: SymPy Matrix (3x3 angular velocity Jacobian)
        det_Jv: SymPy expression for det(J_v)
        J_v_inv: SymPy Matrix (3x3 analytical inverse Jacobian)
        (J_num, v_e, w_e): numerical results if q is given
    """
    # 1. Define symbols
    q1, q2, q3 = sp.symbols('q1 q2 q3', real=True)
    L1, L2, L3 = sp.symbols('L1 L2 L3', real=True)

    # 2. End-effector position vector from FK
    R_ext = L2 + L3 + q3
    px = R_ext * sp.cos(q1) * sp.cos(q2)
    py = R_ext * sp.sin(q1) * sp.cos(q2)
    pz = L1 + R_ext * sp.sin(q2)
    p_vec = sp.Matrix([px, py, pz])
    q_vec = sp.Matrix([q1, q2, q3])

    # 3. Position Jacobian J_v = d(p)/d(q)
    J_v = sp.trigsimp(p_vec.jacobian(q_vec))

    # 4. Angular velocity Jacobian J_w = [z0, z1, 0]
    z0 = sp.Matrix([0, 0, 1])
    z1 = sp.Matrix([sp.sin(q1), -sp.cos(q1), 0])
    z2_prism = sp.Matrix([0, 0, 0])
    J_w = sp.Matrix.hstack(z0, z1, z2_prism)

    # 5. Full 6x3 Geometric Jacobian
    J_geom = sp.Matrix.vstack(J_v, J_w)

    # 6. Singularity Analysis: Determinant of J_v
    det_Jv = sp.trigsimp(J_v.det())

    # 7. Analytical Inverse Jacobian J_v^(-1)
    J_v_inv = sp.trigsimp(J_v.inv())

    if verbose:
        print("="*60)
        print("1. LINEAR VELOCITY JACOBIAN J_v (3x3):")
        print("="*60)
        sp.pprint(J_v)
        print("\n" + "="*60)
        print("2. ANGULAR VELOCITY JACOBIAN J_w (3x3):")
        print("="*60)
        sp.pprint(J_w)
        print("\n" + "="*60)
        print("3. FULL GEOMETRIC JACOBIAN J_0 (6x3):")
        print("="*60)
        sp.pprint(J_geom)
        print("\n" + "="*60)
        print("4. DETERMINANT OF J_v (SINGULARITY FORMULA):")
        print("="*60)
        print(f"det(J_v) = {det_Jv}")
        print("\n" + "="*60)
        print("5. ANALYTIC INVERSE JACOBIAN J_v^(-1) (3x3):")
        print("="*60)
        sp.pprint(J_v_inv)

    # 8. Evaluate numerically if q is given
    if q is not None:
        subs = {q1: q[0], q2: q[1], q3: q[2], L1: L[0], L2: L[1], L3: L[2]}
        J_geom_num = np.array(J_geom.subs(subs)).astype(np.float64)
        J_v_num = np.array(J_v.subs(subs)).astype(np.float64)
        det_num = float(det_Jv.subs(subs))

        res = {"J_geom": J_geom_num, "J_v": J_v_num, "det": det_num}

        if q_dot is not None:
            q_dot = np.asarray(q_dot).reshape(3, 1)
            v_lin = J_v_num @ q_dot
            v_ang = J_geom_num[3:, :] @ q_dot
            res["v_linear"] = v_lin.flatten()
            res["v_angular"] = v_ang.flatten()

        return J_geom, J_v, J_w, det_Jv, J_v_inv, res

    return J_geom, J_v, J_w, det_Jv, J_v_inv


In [29]:
# Run symbolic derivation and verify against Toolbox jacob0
J_geom_sym, J_v_sym, J_w_sym, det_sym, J_v_inv_sym = symbolic_dk(verbose=True)

print("\n" + "#"*60)
print("VERIFICATION: Symbolic Geometric Jacobian vs Robotics Toolbox jacob0")
print("#"*60)

test_jacobian_points = [
    np.array([0.0, 0.0, 0.0]),
    np.array([0.30, 0.40, 0.20]),
    np.array([-0.50, 0.25, 0.35]),
    np.array([np.deg2rad(45), np.deg2rad(30), 0.15])
]

for q_test in test_jacobian_points:
    _, _, _, _, _, res = symbolic_dk(q=q_test, L=(0.50, 0.10, 0.40), verbose=False)
    J_sym_eval = res["J_geom"]
    J_mdh = robot_mdh.jacob0(q_test)
    J_urdf = urdf_robot.jacob0(q_test)

    diff_mdh = np.linalg.norm(J_sym_eval - J_mdh)
    diff_urdf = np.linalg.norm(J_sym_eval - J_urdf)

    print(f"q = {np.round(q_test, 3)} | det(J_v) = {res['det']:.4f}")
    print(f"  Diff(Symbolic vs MDH) : {diff_mdh:.2e}")
    print(f"  Diff(Symbolic vs URDF): {diff_urdf:.2e}")
    print("-" * 50)

# Check singularity detection
print("\nSingularity Check:")
q_sing = np.array([0.0, np.pi / 2, 0.2])  # q2 = 90 deg (arm pointing straight up)
_, _, _, _, _, res_sing = symbolic_dk(q=q_sing, verbose=False)
print(f"At q = {np.round(q_sing, 3)} (q2 = 90 deg): det(J_v) = {res_sing['det']:.6e} -> SINGULAR!")


1. LINEAR VELOCITY JACOBIAN J_v (3x3):
⎡(-L₂ - L₃ - q₃)⋅sin(q₁)⋅cos(q₂)  (-L₂ - L₃ - q₃)⋅sin(q₂)⋅cos(q₁)  cos(q₁)⋅cos ↪
⎢                                                                              ↪
⎢(L₂ + L₃ + q₃)⋅cos(q₁)⋅cos(q₂)   (-L₂ - L₃ - q₃)⋅sin(q₁)⋅sin(q₂)  sin(q₁)⋅cos ↪
⎢                                                                              ↪
⎣               0                     (L₂ + L₃ + q₃)⋅cos(q₂)           sin(q₂) ↪

↪ (q₂)⎤
↪     ⎥
↪ (q₂)⎥
↪     ⎥
↪     ⎦

2. ANGULAR VELOCITY JACOBIAN J_w (3x3):
⎡0  sin(q₁)   0⎤
⎢              ⎥
⎢0  -cos(q₁)  0⎥
⎢              ⎥
⎣1     0      0⎦

3. FULL GEOMETRIC JACOBIAN J_0 (6x3):
⎡(-L₂ - L₃ - q₃)⋅sin(q₁)⋅cos(q₂)  (-L₂ - L₃ - q₃)⋅sin(q₂)⋅cos(q₁)  cos(q₁)⋅cos ↪
⎢                                                                              ↪
⎢(L₂ + L₃ + q₃)⋅cos(q₁)⋅cos(q₂)   (-L₂ - L₃ - q₃)⋅sin(q₁)⋅sin(q₂)  sin(q₁)⋅cos ↪
⎢                                                                              ↪
⎢               0         

## 5. Unified Kinematics Calculator Class & Complete Workflow Demo

Below is the complete, encapsulated `RobotKinematicsCalculator` class. It serves as an all-in-one lab tool providing:
- Forward Kinematics $\mathbf{T}(q)$, $\mathbf{p}(q)$, $\mathbf{R}(q)$
- Inverse Kinematics for any 3D target $[x, y, z]$ (reporting both branches and limit validation)
- Differential Kinematics $\mathbf{J}_0(q)$, $\det(\mathbf{J}_v)$, singularity flags
- Velocity resolution: Joint rates $\dot{\mathbf{q}}$ required for desired Cartesian tip velocity $\dot{\mathbf{p}}$


In [30]:
class RobotKinematicsCalculator:
    """
    Comprehensive Kinematics Calculator for the 3-DOF RRP Manipulator
    supporting symbolic equations, forward/inverse kinematics, Jacobians,
    singularity diagnostics, and resolved-rate velocity control.
    """
    def __init__(self, L1=0.50, L2=0.10, L3=0.40, qlim=None):
        self.L1 = float(L1)
        self.L2 = float(L2)
        self.L3 = float(L3)
        self.qlim = qlim if qlim is not None else np.array([
            [-np.pi/2, np.pi/2],   # q1 limits
            [-np.pi/2, np.pi/2],   # q2 limits
            [0.0, 0.40]            # q3 limits
        ])

    def fk(self, q):
        """Compute Forward Kinematics from joint vector q = [q1, q2, q3]."""
        q1, q2, q3 = q
        R_ext = self.L2 + self.L3 + q3
        x = R_ext * np.cos(q1) * np.cos(q2)
        y = R_ext * np.sin(q1) * np.cos(q2)
        z = self.L1 + R_ext * np.sin(q2)
        pos = np.array([x, y, z])

        R = np.array([
            [-np.cos(q1)*np.sin(q2),  np.sin(q1), np.cos(q1)*np.cos(q2)],
            [-np.sin(q1)*np.sin(q2), -np.cos(q1), np.sin(q1)*np.cos(q2)],
            [ np.cos(q2),            0.0,         np.sin(q2)]
        ])

        T = np.eye(4)
        T[:3, :3] = R
        T[:3, 3] = pos
        return {"T": T, "position": pos, "rotation": R}

    def ik(self, target_pos):
        """Compute Inverse Kinematics for target_pos = [x, y, z]. Returns both branches."""
        x, y, z = target_pos
        dx = x
        dy = y
        dz = z - self.L1

        rho_sq = dx**2 + dy**2 + dz**2
        R_ext = np.sqrt(rho_sq)
        q3 = R_ext - self.L2 - self.L3

        # Branch 1 (Forward)
        q1_b1 = np.arctan2(dy, dx)
        r_xy = np.sqrt(dx**2 + dy**2)
        q2_b1 = np.arctan2(dz, r_xy)
        b1 = np.array([q1_b1, q2_b1, q3])

        # Branch 2 (Backward / Flipped)
        q1_b2 = np.arctan2(-dy, -dx)
        q2_b2 = np.arctan2(dz, -r_xy)
        b2 = np.array([q1_b2, q2_b2, q3])

        b1_valid = bool(np.all((b1 >= self.qlim[:, 0]) & (b1 <= self.qlim[:, 1])))
        b2_valid = bool(np.all((b2 >= self.qlim[:, 0]) & (b2 <= self.qlim[:, 1])))

        return {
            "branch1": {"q": b1, "valid": b1_valid},
            "branch2": {"q": b2, "valid": b2_valid},
            "recommended": b1 if b1_valid else (b2 if b2_valid else None)
        }

    def jacobian(self, q):
        """Compute the 6x3 geometric Jacobian, 3x3 position Jacobian, and determinant."""
        q1, q2, q3 = q
        R_ext = self.L2 + self.L3 + q3

        J_v = np.array([
            [-R_ext*np.sin(q1)*np.cos(q2), -R_ext*np.cos(q1)*np.sin(q2), np.cos(q1)*np.cos(q2)],
            [ R_ext*np.cos(q1)*np.cos(q2), -R_ext*np.sin(q1)*np.sin(q2), np.sin(q1)*np.cos(q2)],
            [ 0.0,                          R_ext*np.cos(q2),            np.sin(q2)]
        ])

        J_w = np.array([
            [0.0,  np.sin(q1), 0.0],
            [0.0, -np.cos(q1), 0.0],
            [1.0,  0.0,        0.0]
        ])

        J_geom = np.vstack([J_v, J_w])
        det_Jv = (R_ext**2) * np.cos(q2)
        is_singular = abs(det_Jv) < 1e-4

        return {
            "J_geom": J_geom,
            "J_v": J_v,
            "J_w": J_w,
            "det": det_Jv,
            "is_singular": is_singular
        }

    def resolve_rates(self, q, desired_lin_vel):
        """Given desired linear velocity [vx, vy, vz], solve for joint rates q_dot = J_v^(-1) * v."""
        j_data = self.jacobian(q)
        if j_data["is_singular"]:
            raise ValueError(f"Robot is near a kinematic singularity! det(J_v) = {j_data['det']:.4e}")
        J_v_inv = np.linalg.inv(j_data["J_v"])
        q_dot = J_v_inv @ np.asarray(desired_lin_vel)
        return q_dot

# Instantiate Calculator
calc = RobotKinematicsCalculator(L1=0.50, L2=0.10, L3=0.40)

# 1. Test FK
fk_res = calc.fk([np.deg2rad(30), np.deg2rad(15), 0.20])
print("=== 1. FORWARD KINEMATICS TEST ===")
print("End-Effector Position:", np.round(fk_res["position"], 4))

# 2. Test IK
target_M = np.array([0.45, -0.20, 0.9501])
ik_res = calc.ik(target_M)
print("\n=== 2. INVERSE KINEMATICS TEST ===")
print("Target Position:", target_M)
print("Branch 1 q:", np.round(ik_res["branch1"]["q"], 4), "| Valid:", ik_res["branch1"]["valid"])
print("Branch 2 q:", np.round(ik_res["branch2"]["q"], 4), "| Valid:", ik_res["branch2"]["valid"])

# 3. Test DK & Velocity Resolution
q_op = ik_res["recommended"]
v_des = np.array([0.05, 0.0, -0.02])  # 5 cm/s in x, -2 cm/s in z
q_dot = calc.resolve_rates(q_op, v_des)

print("\n=== 3. DIFFERENTIAL KINEMATICS TEST ===")
print("Operating Joint Configuration q:", np.round(q_op, 4))
print("Desired Tip Linear Velocity   v:", v_des)
print("Required Joint Velocities q_dot:", np.round(q_dot, 4))

# Re-multiply J_v * q_dot to confirm
J_v = calc.jacobian(q_op)["J_v"]
v_achieved = J_v @ q_dot
print("Achieved Velocity (J_v * q_dot):", np.round(v_achieved, 4))
print("Tracking Error                 :", np.linalg.norm(v_achieved - v_des))


=== 1. FORWARD KINEMATICS TEST ===
End-Effector Position: [0.5856 0.3381 0.6812]

=== 2. INVERSE KINEMATICS TEST ===
Target Position: [ 0.45   -0.2     0.9501]
Branch 1 q: [-0.4182  0.7405  0.1672] | Valid: True
Branch 2 q: [2.7234 2.4011 0.1672] | Valid: False

=== 3. DIFFERENTIAL KINEMATICS TEST ===
Operating Joint Configuration q: [-0.4182  0.7405  0.1672]
Desired Tip Linear Velocity   v: [ 0.05  0.   -0.02]
Required Joint Velocities q_dot: [ 0.0412 -0.0683  0.0202]
Achieved Velocity (J_v * q_dot): [ 0.05  0.   -0.02]
Tracking Error                 : 1.0399577078266507e-17


## 6. Summary & Analytical Formulations Reference

| Concept | Closed-Form Symbolic Expression | Notes |
|:---|:---|:---|
| **Forward Position $\mathbf{p}(q)$** | $\begin{bmatrix} (L_2+L_3+q_3)\cos(q_1)\cos(q_2) \\ (L_2+L_3+q_3)\sin(q_1)\cos(q_2) \\ L_1 + (L_2+L_3+q_3)\sin(q_2) \end{bmatrix}$ | Matches MDH & URDF ($< 10^{-15}$ error) |
| **Inverse $q_3$** | $q_3 = \sqrt{x^2 + y^2 + (z - L_1)^2} - L_2 - L_3$ | Unique physical positive root |
| **Inverse $q_1$** | $q_1 = \text{atan2}(\pm y, \pm x)$ | Branch 1: $+$, Branch 2: $-$ ($\pm \pi$) |
| **Inverse $q_2$** | $q_2 = \text{atan2}(z - L_1, \pm \sqrt{x^2 + y^2})$ | Branch 1: $+r_{xy}$, Branch 2: $-r_{xy}$ |
| **Linear Jacobian $\mathbf{J}_v(q)$** | $\begin{bmatrix} -R\sin(q_1)\cos(q_2) & -R\cos(q_1)\sin(q_2) & \cos(q_1)\cos(q_2) \\ R\cos(q_1)\cos(q_2) & -R\sin(q_1)\sin(q_2) & \sin(q_1)\cos(q_2) \\ 0 & R\cos(q_2) & \sin(q_2) \end{bmatrix}$ | Where $R = L_2 + L_3 + q_3$ |
| **Angular Jacobian $\mathbf{J}_\omega(q)$** | $\begin{bmatrix} 0 & \sin(q_1) & 0 \\ 0 & -\cos(q_1) & 0 \\ 1 & 0 & 0 \end{bmatrix}$ | Base rotation axes $\mathbf{z}_0, \mathbf{z}_1, \mathbf{0}$ |
| **Singularity $\det(\mathbf{J}_v)$** | $\det(\mathbf{J}_v) = (L_2 + L_3 + q_3)^2 \cos(q_2)$ | Singular at $q_2 = \pm 90^\circ$ |
| **Inverse Velocity $\dot{\mathbf{q}}$** | $\dot{\mathbf{q}} = \mathbf{J}_v^{-1} \dot{\mathbf{p}}$ | Resolved rate motion control |


## 7. Integrated Pipeline: From FK to IK to DK with Robotics Toolbox (RTB)

In robot motion planning and control, **Forward Kinematics (FK)**, **Inverse Kinematics (IK)**, and **Differential Kinematics (DK)** form a continuous sequential pipeline:

```
+-----------------------------------------------------------------------------------+
|                           THE KINEMATICS PIPELINE                                 |
+-----------------------------------------------------------------------------------+
|  1. FORWARD KINEMATICS (FK)                                                       |
|     "Where is my end-effector currently located?"                                 |
|     Input: q = [q1, q2, q3]  --->  Output: T_0E, p_e = [x, y, z]                  |
|     Toolbox Key Function: robot.fkine(q)                                          |
+-----------------------------------------------------------------------------------+
|  2. INVERSE KINEMATICS (IK)                                                       |
|     "What joint configuration reaches the target object M?"                       |
|     Input: Target M = [x, y, z]  --->  Output: q_goal (Branch 1 / Branch 2)       |
|     Toolbox Key Function: robot.ikine_LM(SE3(*M), mask=[1,1,1,0,0,0])             |
+-----------------------------------------------------------------------------------+
|  3. DIFFERENTIAL KINEMATICS (DK)                                                  |
|     "How fast and in what direction must the joints move to trace a path?"        |
|     Velocity Mapping: v_e = J_v(q) * q_dot                                        |
|     Resolved-Rate Motion: q_dot = J_v^(-1) * v_desired                            |
|     Toolbox Key Functions:                                                        |
|       - robot.jacob0(q)                    : 6xn Geometric Jacobian in base frame |
|       - robot.jacobe(q)                    : 6xn Geometric Jacobian in tool frame |
|       - robot.manipulability(q, axes='trans'): Translational manipulability measure|
|       - np.linalg.pinv(J_v) / inv(J_v)     : Resolved-rate inverse velocity solver |
+-----------------------------------------------------------------------------------+
```

### Key RTB Functions for Differential Kinematics:
1. `robot.jacob0(q)`: Calculates $\mathbf{J}_0(q) \in \mathbb{R}^{6 \times n}$ with respect to the world/base frame $\{0\}$. For our 3-DOF robot, `robot.jacob0(q)[:3, :]` extracts the $3 \times 3$ linear velocity Jacobian $\mathbf{J}_v(q)$.
2. `robot.jacobe(q)`: Calculates $\mathbf{J}_e(q) \in \mathbb{R}^{6 \times n}$ with respect to the tool frame $\{E\}$.
3. `robot.manipulability(q, axes='trans')`: Computes Yoshikawa's translational manipulability $w = \sqrt{\det(\mathbf{J}_v \mathbf{J}_v^T)} = |\det(\mathbf{J}_v)|$. *(Note: For a 3-DOF robot, specifying `axes='trans'` is essential because the full 6-DOF manipulability is zero due to rank $\le 3 < 6$.)*
4. `np.linalg.pinv(J_v)` / `np.linalg.inv(J_v)`: Solves the inverse differential kinematics problem $\dot{\mathbf{q}} = \mathbf{J}_v^{-1} \dot{\mathbf{p}}$ for path tracking.


In [31]:
# ==============================================================================
# SECTION 7: FULL WORKFLOW DEMONSTRATION (FK -> IK -> DK)
# ==============================================================================
print("=================================================================")
print("STEP 1: FORWARD KINEMATICS (FK)")
print("=================================================================")
# Current robot state at Home
q_current = np.array([0.0, 0.0, 0.0])
T_current = urdf_robot.fkine(q_current)
p_current = T_current.t
print(f"Current Joint Angles q: {q_current}")
print(f"Current Tip Position  : {np.round(p_current, 4)} m")

print("\n=================================================================")
print("STEP 2: INVERSE KINEMATICS (IK)")
print("=================================================================")
# Given target from perception in base frame:
target_M = np.array([0.45, -0.20, 0.9501])
print(f"Goal Target Position M : {target_M}")

# 2.1 Symbolic IK Solution (Branch 1 & Branch 2)
_, b1, b2 = symbolic_ik(M=target_M, check_limits=False, verbose=False)
qlim = urdf_robot.qlim
b1_ok = np.all((b1 >= qlim[0]) & (b1 <= qlim[1]))
q_goal = b1 if b1_ok else b2
print(f"Symbolic IK Solution q_goal : {np.round(q_goal, 4)} (Within limits: {b1_ok})")

# 2.2 Numerical IK with RTB ikine_LM
sol_num = urdf_robot.ikine_LM(SE3(*target_M), q0=q_current, mask=[1, 1, 1, 0, 0, 0], joint_limits=True)
print(f"Toolbox ikine_LM Solution   : {np.round(sol_num.q, 4)} (Success: {sol_num.success})")

# Verify error
pos_reached = urdf_robot.fkine(q_goal).t
print(f"Reached Position with q_goal: {np.round(pos_reached, 4)}")
print(f"Target Reaching Error       : {np.linalg.norm(pos_reached - target_M):.2e} m")

print("\n=================================================================")
print("STEP 3: DIFFERENTIAL KINEMATICS (DK) AT q_goal")
print("=================================================================")
# 3.1 Base-frame Geometric Jacobian: robot.jacob0(q)
J0 = urdf_robot.jacob0(q_goal)
J_v = J0[:3, :]  # Linear velocity part (3x3)
J_w = J0[3:, :]  # Angular velocity part (3x3)

# 3.2 Tool-frame Geometric Jacobian: robot.jacobe(q)
Je = urdf_robot.jacobe(q_goal)

print("Base-Frame Linear Jacobian J_v (3x3):")
print(np.round(J_v, 4))
print("\nTool-Frame Linear Jacobian J_v_e (3x3):")
print(np.round(Je[:3, :], 4))

# 3.3 Singularity & Manipulability
det_Jv = np.linalg.det(J_v)
cond_Jv = np.linalg.cond(J_v)
w_trans = urdf_robot.manipulability(q_goal, axes='trans')

print(f"\nDeterminant det(J_v)          : {det_Jv:.4f}")
print(f"Condition Number cond(J_v)    : {cond_Jv:.4f} (Close to 1 is well-conditioned)")
print(f"RTB Manipulability (trans)    : {w_trans:.4f} (Equals |det(J_v)|)")

print("\n=================================================================")
print("STEP 4: RESOLVED-RATE TRAJECTORY MOTION (DK VELOCITY TRACKING)")
print("=================================================================")
# Move the end-effector downward by 5 cm at constant velocity vz = -0.05 m/s
v_cmd = np.array([0.0, 0.0, -0.05])  # Cartesian velocity [vx, vy, vz]
dt = 0.02                            # 50 Hz control loop (20 ms)
total_steps = 25                     # 0.5 seconds duration

q_sim = q_goal.copy()
trajectory_errors = []

print(f"Commanded Cartesian Velocity: {v_cmd} m/s over {total_steps * dt:.2f} s")
print("Step | Joint Rates q_dot [rad/s, rad/s, m/s] | Tip z [m] | Tracking Err [m]")
print("-" * 75)

for step in range(total_steps):
    # Compute instantaneous Jacobian
    J_v_k = urdf_robot.jacob0(q_sim)[:3, :]
    
    # Resolved-rate inverse: q_dot = J_v^(-1) * v_cmd
    q_dot = np.linalg.inv(J_v_k) @ v_cmd
    
    # Numerical Euler integration: q_{k+1} = q_k + q_dot * dt
    q_sim = q_sim + q_dot * dt
    
    # Measure actual FK tip position
    p_actual = urdf_robot.fkine(q_sim).t
    p_ideal = target_M + v_cmd * (step + 1) * dt
    err = np.linalg.norm(p_actual - p_ideal)
    trajectory_errors.append(err)
    
    if step % 5 == 0 or step == total_steps - 1:
        print(f"{step:4d} | {np.round(q_dot, 4)} | {p_actual[2]:.4f}   | {err:.2e}")

print("-" * 75)
print(f"-> Maximum Trajectory Tracking Deviation: {max(trajectory_errors):.2e} m")
print("-> Pipeline Complete: Successfully integrated FK -> IK -> DK!")


STEP 1: FORWARD KINEMATICS (FK)
Current Joint Angles q: [0. 0. 0.]
Current Tip Position  : [ 0.5 -0.   0.5] m

STEP 2: INVERSE KINEMATICS (IK)
Goal Target Position M : [ 0.45   -0.2     0.9501]
Symbolic IK Solution q_goal : [-0.4182  0.7405  0.1672] (Within limits: True)
Toolbox ikine_LM Solution   : [-0.4173  0.7395  0.1664] (Success: True)
Reached Position with q_goal: [ 0.45   -0.2     0.9501]
Target Reaching Error       : 1.24e-16 m

STEP 3: DIFFERENTIAL KINEMATICS (DK) AT q_goal
Base-Frame Linear Jacobian J_v (3x3):
[[ 0.2    -0.4113  0.6745]
 [ 0.45    0.1828 -0.2998]
 [-0.      0.4924  0.6747]]

Tool-Frame Linear Jacobian J_v_e (3x3):
[[ 0.      0.6672  0.    ]
 [-0.4924  0.      0.    ]
 [-0.      0.      1.    ]]

Determinant det(J_v)          : 0.3285
Condition Number cond(J_v)    : 2.0307 (Close to 1 is well-conditioned)
RTB Manipulability (trans)    : 0.3285 (Equals |det(J_v)|)

STEP 4: RESOLVED-RATE TRAJECTORY MOTION (DK VELOCITY TRACKING)
Commanded Cartesian Velocity: [ 0

### Summary of RTB Functions Used in Section 7

| Step | Goal | RTB / NumPy Function | Description |
|:---|:---|:---|:---|
| **1. FK** | Current pose calculation | `robot.fkine(q)` | Evaluates $4 \times 4$ homogeneous transform $\mathbf{T}_{0E}(q)$ |
| **2. IK** | Numerical target solving | `robot.ikine_LM(SE3(*M))` | Solves $\mathbf{q}$ numerically using Levenberg-Marquardt with joint limits |
| **3. DK** | Base-frame Jacobian | `robot.jacob0(q)` | Computes $\mathbf{J}_0(q) \in \mathbb{R}^{6 \times n}$; `[:3, :]` gives $\mathbf{J}_v(q)$ |
| **3. DK** | Tool-frame Jacobian | `robot.jacobe(q)` | Computes $\mathbf{J}_e(q) \in \mathbb{R}^{6 \times n}$ relative to end-effector frame |
| **3. DK** | Manipulability index | `robot.manipulability(q, axes='trans')` | Calculates translational manipulability $|\det(\mathbf{J}_v)|$ |
| **4. DK** | Resolved-rate control | `np.linalg.inv(J_v) @ v_cmd` | Computes $\dot{\mathbf{q}}$ to track arbitrary task-space velocity $\mathbf{v}$ |
